# Projet E-PRTR / IED — Notebook de guidage

**Pour toi, Olivier** : ce notebook t'explique, pas à pas, comment recadrer ton projet de façon propre en data engineering, sans te perdre dans les problèmes de granularité et de jointure.


## 1. Ce que tu as bien vu

Tu as repéré les **deux vrais problèmes** de ton projet :

1. **les tables ne sont pas au même niveau de granularité** ;
2. **la clé de jointure n'est pas directe**.

C'est une très bonne observation. En data engineering, c'est exactement le genre de problème qu'on doit détecter **avant** de faire des jointures.

Ton projet reste intéressant, mais il faut le reformuler proprement :

> **Étudier l'évolution environnementale de sites industriels belges entre 2016 et 2024, en construisant un modèle multi-grain et un bridge entre les LCP, les installations et les facilities.**

Le mot-clé ici, c'est : **multi-grain**.



## 2. Le vrai problème de grain

### Table `F1_4_Air_Releases_Facilities`
Grain :
- **1 facility × 1 année × 1 polluant**

### Table `F5_2_LCP_Energy_Emissions`
Grain :
- **1 LCP × 1 année × 1 type de mesure**

Le `featureType` mélange plusieurs natures de lignes :
- combustibles (`NaturalGas`, `Coal`, `Biomass`, etc.)
- polluants (`NOX`, `SO2`, `DUST`)
- caractéristiques techniques (`LCPCharacteristics`)

### Table `F6_1_IED_Installations`
D'après ton notebook, cette table sert de **pont** entre :
- l'installation / installation part
- la facility parente

Donc non, tu ne peux **pas** joindre directement `F1_4` et `F5_2`.

Si tu fais ça, tu vas mélanger :
- des lignes au niveau **facility**
- avec des lignes au niveau **LCP**
- et parfois plusieurs lignes par année et par installation part

Résultat : tu risques de **dupliquer** des volumes, de **surcomptabiliser** des valeurs, ou de produire un indicateur faux.



## 3. Mon conseil principal

### Tu dois choisir un **grain cible** pour ta table analytique finale

Le plus logique pour ton projet est :

> **facility × year**

Pourquoi ?

Parce que :
- ton CO2 dans `F1_4` est au niveau **facility**
- plusieurs enrichissements utiles (`F2_4`, `F3_2`) sont aussi au niveau **facility**
- c'est plus simple pour un premier projet de data engineering

Ensuite, tu **remontes** les autres tables vers ce grain.

Donc la logique devient :

```text
RAW tables
   ↓
Silver F1 : facility × year
Silver F5 : LCP × year
Silver bridge : LCP → installation → facility
   ↓
Gold : facility × year
```



## 4. Ce que tu ne dois pas faire

### Erreur classique n°1
Joindre :

- `FacilityInspireId`
avec
- `LCPInspireId`

Ce n'est **pas** la même entité.

### Erreur classique n°2
Calculer directement :

```text
CO2 facility / featureValue
```

sans filtrer `featureType`.

Dans `F5_2`, toutes les lignes ne sont **pas** des consommations énergétiques.

### Erreur classique n°3
Faire une étude sur **2007–2024** en croisant `F1_4` et `F5_2`.

Pourquoi ?
Parce que `F1_4` Belgique CO2 couvre **2007–2024**, alors que `F5_2` Belgique couvre surtout **2016–2024**.

Donc pour ton **modèle croisé principal**, travaille sur :

> **2016–2024**


In [2]:

import pandas as pd
import numpy as np

# Mets les bons chemins si nécessaire
f1_path = "./User friendly .csv files/F1_4_Air_Releases_Facilities.csv"
f5_path = "./User friendly .csv files/F5_2_LCP_Energy_Emissions.csv"
f2_4_path = "./User friendly .csv files/F2_4_Water_Releases_Facilities.csv"
f3_2_path = "./User friendly .csv files/F3_2_Transfers_Facilities.csv"

df_f1 = pd.read_csv(f1_path, low_memory=False)
df_f5 = pd.read_csv(f5_path, low_memory=False)
df_f2_4 = pd.read_csv(f2_4_path, low_memory=False)
df_f3_2 = pd.read_csv(f3_2_path, low_memory=False)

print("F1_4 shape :", df_f1.shape)
print("F5_2 shape :", df_f5.shape)
print("F2_4 shape :", df_f2_4.shape)
print("F3_2 shape :", df_f3_2.shape)


F1_4 shape : (372206, 16)
F5_2 shape : (401072, 14)
F2_4 shape : (254156, 16)
F3_2 shape : (65951, 15)


In [10]:

# Focus Belgique
be_f1 = df_f1[(df_f1["countryName"] == "Belgium") & (df_f1["Pollutant"] == "Carbon dioxide (CO2)")].copy()
be_f5 = df_f5[df_f5["countryName"] == "Belgium"].copy()
be_f2_4 = df_f2_4[df_f2_4["countryName"] == "Belgium"].copy()
be_f3_2 = df_f3_2[df_f3_2["countryName"] == "Belgium"].copy()

print("Belgique - F1 CO2 :", be_f1.shape)
print("Belgique - F5 :", be_f5.shape)
print("Belgique - F2_4 :", be_f2_4.shape)
print("Belgique - F3_2 :", be_f3_2.shape)

print("\nAnnées F1 CO2 :", sorted(be_f1["reportingYear"].dropna().unique()))
print("Années F5 :", sorted(be_f5["reportingYear"].dropna().unique()))
print("Années F2_4 :", sorted(be_f2_4["reportingYear"].dropna().unique()))
print("Années F3_2 :", sorted(be_f3_2["reportingYear"].dropna().unique()))


Belgique - F1 CO2 : (1270, 16)
Belgique - F5 : (10521, 14)
Belgique - F2_4 : (7044, 16)
Belgique - F3_2 : (1175, 15)

Années F1 CO2 : [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Années F5 : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Années F2_4 : [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Années F3_2 : [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int


## 5. Construis d'abord des tables Silver propres

### Silver 1 — CO2 facility-year

Garde seulement :
- Belgique
- CO2
- années 2016 à 2024

Tu dois obtenir une table au grain :

> **FacilityInspireId × reportingYear**


In [ ]:

silver_f1_facility_co2 = (
    be_f1.loc[be_f1["reportingYear"].between(2016, 2024),
              ["FacilityInspireId", "facilityName", "city", "Longitude", "Latitude",
               "EPRTR_SectorCode", "EPRTR_SectorName", "EPRTRAnnexIMainActivity",
               "reportingYear", "Releases"]]
    .rename(columns={"Releases": "facility_co2"})
    .groupby(
        ["FacilityInspireId", "facilityName", "city", "Longitude", "Latitude",
         "EPRTR_SectorCode", "EPRTR_SectorName", "EPRTRAnnexIMainActivity", "reportingYear"],
        as_index=False
    )["facility_co2"].sum()
)

silver_f1_facility_co2.head()


,FacilityInspireId,facilityName,city,Longitude,Latitude,EPRTR_SectorCode,EPRTR_SectorName,EPRTRAnnexIMainActivity,reportingYear,facility_co2
0,BE.BRU/100010004.FACILITY,BRUXELLES ENERGIE,Bruxelles,4.38037,50.8831,5.0,Waste and wastewater management,5(b),2016,492000000.0
1,BE.BRU/100010004.FACILITY,BRUXELLES ENERGIE,Bruxelles,4.38037,50.8831,5.0,Waste and wastewater management,5(b),2017,427000000.0
2,BE.BRU/100010004.FACILITY,BRUXELLES ENERGIE,Bruxelles,4.38037,50.8831,5.0,Waste and wastewater management,5(b),2018,485000000.0
3,BE.BRU/100010004.FACILITY,BRUXELLES ENERGIE,Bruxelles,4.38037,50.8831,5.0,Waste and wastewater management,5(b),2019,625000000.0
4,BE.BRU/100010004.FACILITY,BRUXELLES ENERGIE,Bruxelles,4.38037,50.8831,5.0,Waste and wastewater management,5(b),2020,442000000.0



### Silver 2 — Énergie LCP par année

Ici, tu dois **filtrer les lignes pertinentes**.

Dans `F5_2`, toutes les lignes ne correspondent pas à de l'énergie consommée.  
Pour commencer simplement, je te conseille de garder les combustibles :

- `NaturalGas`
- `Coal`
- `Lignite`
- `Biomass`
- `OtherSolidFuels`
- `OtherGases`
- `LiquidFuels`
- `Peat`

Puis tu agrèges au grain :

> **LCPInspireId × reportingYear**


In [5]:

fuel_feature_types = [
    "NaturalGas", "Coal", "Lignite", "Biomass",
    "OtherSolidFuels", "OtherGases", "LiquidFuels", "Peat"
]

silver_f5_lcp_energy = (
    be_f5.loc[
        be_f5["reportingYear"].between(2016, 2024)
        & be_f5["featureType"].isin(fuel_feature_types),
        ["LCPInspireId", "installationPartName", "City_Of_Facility",
         "Longitude", "Latitude", "reportingYear", "featureType", "unit", "featureValue"]
    ]
    .rename(columns={"featureValue": "energy_value"})
    .copy()
)

silver_f5_lcp_energy.head()


,LCPInspireId,installationPartName,City_Of_Facility,Longitude,Latitude,reportingYear,featureType,unit,energy_value
102999,https://data.ied_registry.omgeving.vlaanderen....,TOTALENERGIES REFINERY ANTWERP_LCP 6,Antwerpen,4.321080,51.267370,2016,NaturalGas,TJ,0.0
103066,https://data.ied_registry.omgeving.vlaanderen....,ZANDVLIET POWER - TERREIN BASF,Antwerpen,4.266880,51.368860,2020,LiquidFuels,TJ,0.0
104197,BE.WA/115010101.PART,Cierreux I (Turbo jet back-up),Bovigny,5.912952,50.250020,2019,OtherGases,TJ,0.0
104303,BE.WA/049010702.PART,Solvay GT1A,Jemeppe-Sur-Sambre,4.662693,50.447224,2016,Biomass,TJ,0.0
104305,https://data.ied_registry.omgeving.vlaanderen....,LUMINUS Ham_LCP 2,Gent,3.736470,51.059740,2017,Lignite,TJ,0.0


In [6]:

# Agrégation simple au niveau LCP × année
silver_f5_lcp_energy_total = (
    silver_f5_lcp_energy
    .groupby(["LCPInspireId", "installationPartName", "City_Of_Facility",
              "Longitude", "Latitude", "reportingYear"], as_index=False)["energy_value"]
    .sum()
    .rename(columns={"energy_value": "lcp_energy_total"})
)

silver_f5_lcp_energy_total.head()


,LCPInspireId,installationPartName,City_Of_Facility,Longitude,Latitude,reportingYear,lcp_energy_total
0,BE.BRU/100010002.PART,TURBOJET BUDA,Bruxelles,4.4112,50.9067,2016,4.15
1,BE.BRU/100010002.PART,TURBOJET BUDA,Bruxelles,4.4112,50.9067,2017,21.94
2,BE.BRU/100010002.PART,TURBOJET BUDA,Bruxelles,4.4112,50.9067,2018,6.91
3,BE.BRU/100010002.PART,TURBOJET BUDA,Bruxelles,4.4112,50.9067,2019,2.96
4,BE.BRU/100010002.PART,TURBOJET BUDA,Bruxelles,4.4112,50.9067,2020,1.21



## 6. Le problème de clé : comment le traiter proprement

D'après ton notebook, la table clé est :

> `F6_1_IED_Installations`

C'est elle qui doit te permettre de passer de :
- `InstallationInspireId`
- vers `parent_facilityInspireId`

### Très important
Tu dois construire une **table bridge**.

Par exemple :

```text
bridge_lcp_facility
- LCPInspireId
- InstallationInspireId
- FacilityInspireId
- mapping_method
- confidence_score
- validation_status
```

### Règle d'or
Si tu n'as pas une correspondance claire entre `LCPInspireId` et `InstallationInspireId`, **n'invente pas** une jointure.

Dans ce cas :
- soit tu récupères la vraie table de mapping,
- soit tu fais un mapping semi-automatique basé sur
  - nom,
  - ville,
  - coordonnées,
- puis tu **documentes** le niveau de confiance.



## 7. Si tu n'as pas encore réussi la jointure LCP → installation → facility

Voici mon conseil très concret :

### Option 1 — la meilleure
Retrouve et charge la table `F6_1_IED_Installations` dans ton projet.

### Option 2 — acceptable pour un débutant, si tu documentes bien
Construis un bridge provisoire avec :
- nom normalisé,
- ville,
- distance géographique,
- contrôle manuel des cas ambigus.

### Option 3 — à éviter
Faire croire que `LCPInspireId == FacilityInspireId`.

Ça, il ne faut pas le faire.


In [ ]:

# Exemple de normalisation minimale des noms
def normalize_text(x):
    if pd.isna(x):
        return None
    x = str(x).lower().strip()
    for ch in ["-", "/", "(", ")", ",", ".", ";", ":", "'"]:
        x = x.replace(ch, " ")
    x = " ".join(x.split())
    return x

silver_f1_facility_co2["facility_name_norm"] = silver_f1_facility_co2["facilityName"].map(normalize_text)
silver_f5_lcp_energy_total["lcp_name_norm"] = silver_f5_lcp_energy_total["installationPartName"].map(normalize_text)

silver_f1_facility_co2[["facilityName", "facility_name_norm"]].head()



## 8. Une fois le bridge prêt, tu remontes au grain facility-year

Quand tu auras la table de bridge, ta logique sera :

1. `silver_f5_lcp_energy_total`
2. jointure avec `bridge_lcp_facility`
3. agrégation au niveau **FacilityInspireId × reportingYear**
4. jointure avec `silver_f1_facility_co2`

Et là seulement, tu construis ta table Gold.


In [ ]:

# PSEUDO-CODE : à adapter quand ton bridge sera prêt

# bridge_lcp_facility = pd.read_csv("ton_bridge.csv")

# silver_f5_facility_energy = (
#     silver_f5_lcp_energy_total
#     .merge(bridge_lcp_facility[["LCPInspireId", "FacilityInspireId"]],
#            on="LCPInspireId", how="left")
#     .groupby(["FacilityInspireId", "reportingYear"], as_index=False)["lcp_energy_total"]
#     .sum()
# )

# gold_facility_year = silver_f1_facility_co2.merge(
#     silver_f5_facility_energy,
#     on=["FacilityInspireId", "reportingYear"],
#     how="left"
# )

# gold_facility_year["carbon_intensity_proxy"] = (
#     gold_facility_year["facility_co2"] / gold_facility_year["lcp_energy_total"]
# )

# gold_facility_year.head()



## 9. Attention : ton indicateur est un **proxy**

Si tu calcules :

```text
CO2 facility / énergie LCP
```

tu dois l'appeler :

> **proxy d'intensité carbonique**

Pourquoi ?
Parce que :
- le CO2 vient du **site entier**
- l'énergie ne couvre que les **LCP**

Donc ton ratio peut être utile, mais il n'est pas parfaitement homogène sur le plan métier.

### Formulation correcte
Tu peux écrire dans ton projet :

> L'indicateur construit est un proxy d'intensité carbonique au niveau facility-year, car le numérateur provient des émissions du site entier alors que le dénominateur provient des consommations énergétiques des Large Combustion Plants rattachées au site.



## 10. Enrichissements que je te conseille

Une fois ton problème de grain réglé, tu peux faire un vrai projet d'analyse environnementale.

### Enrichissement A — rejets dans l'eau
Table :
- `F2_4_Water_Releases_Facilities`

Pourquoi ?
Parce qu'elle est déjà au niveau **facility × year × pollutant**.  
Donc elle se marie beaucoup mieux avec `F1_4`.

### Enrichissement B — transferts
Table :
- `F3_2_Transfers_Facilities`

Pourquoi ?
Parce que tu peux construire un profil environnemental plus riche :
- CO2 air
- polluants eau
- transferts hors site

### Enrichissement C — benchmark secteur
Tables :
- `F1_2_Air_Releases_Sector`
- `F1_3_Air_Releases_AnnexIActivity`

Pourquoi ?
Parce que tu peux comparer chaque site :
- à son secteur
- à son activité réglementaire

### Enrichissement D — benchmark national
Tables :
- `F1_1_Air_Releases_National`
- `F5_1_LCP_Energy_Emissions_National`

Pourquoi ?
Parce que tu peux situer la Belgique dans une tendance plus large.


In [7]:

# Exemple simple : enrichissement eau au niveau facility-year

water_facility_year = (
    be_f2_4.loc[be_f2_4["reportingYear"].between(2016, 2024),
                ["FacilityInspireId", "reportingYear", "Pollutant", "Releases"]]
    .groupby(["FacilityInspireId", "reportingYear"], as_index=False)["Releases"]
    .sum()
    .rename(columns={"Releases": "water_releases_total"})
)

water_facility_year.head()


,FacilityInspireId,reportingYear,water_releases_total
0,BE.BRU/100010015.FACILITY,2016,4347775.58
1,BE.BRU/100010015.FACILITY,2017,3825309.00
2,BE.BRU/100010015.FACILITY,2018,4683411.00
3,BE.BRU/100010015.FACILITY,2019,3965451.00
4,BE.BRU/100010015.FACILITY,2020,3874096.00


In [8]:

# Exemple simple : enrichissement transferts au niveau facility-year

transfers_facility_year = (
    be_f3_2.loc[be_f3_2["reportingYear"].between(2016, 2024),
                ["FacilityInspireId", "reportingYear", "transfers"]]
    .groupby(["FacilityInspireId", "reportingYear"], as_index=False)["transfers"]
    .sum()
    .rename(columns={"transfers": "transfers_total"})
)

transfers_facility_year.head()


,FacilityInspireId,reportingYear,transfers_total
0,BE.BRU/100010003.FACILITY,2016,67.0
1,BE.BRU/100010003.FACILITY,2018,61.8
2,BE.BRU/100010006.FACILITY,2016,22.8
3,BE.BRU/100010009.FACILITY,2016,85900.0
4,BE.BRU/100010009.FACILITY,2017,85900.0



## 11. Ce que je te conseille comme table Gold finale

Nom possible :

> `gold_facility_year_environment_profile`

Grain :

> **FacilityInspireId × reportingYear**

Colonnes possibles :
- `facility_co2`
- `lcp_energy_total`
- `carbon_intensity_proxy`
- `water_releases_total`
- `transfers_total`
- `EPRTR_SectorName`
- `EPRTRAnnexIMainActivity`
- `city`
- `Longitude`
- `Latitude`
- `mapping_confidence`
- `nb_lcp_linked`

Avec ça, tu peux faire de la BI, de la cartographie, du benchmark et même du clustering.



## 12. Analyses intéressantes que tu peux faire

### Analyse 1 — trajectoire des sites
- quels sites diminuent leur CO2 ?
- lesquels restent très intenses ?
- lesquels ont un profil instable ?

### Analyse 2 — benchmark sectoriel
- un site fait-il mieux ou moins bien que son secteur ?
- son intensité proxy baisse-t-elle plus vite que la moyenne ?

### Analyse 3 — analyse multi-pression
- un site baisse-t-il le CO2 mais augmente-t-il ses rejets eau ?
- certains sites déplacent-ils la pression environnementale d'un milieu vers un autre ?

### Analyse 4 — cartographie
- où sont les hotspots ?
- quelles régions concentrent les plus gros volumes ?
- les profils changent-ils dans le temps ?



## 13. Ton plan de travail conseillé

### Étape 1
Valider le grain de chaque table.

### Étape 2
Construire `silver_f1_facility_co2`.

### Étape 3
Construire `silver_f5_lcp_energy_total`.

### Étape 4
Construire ou récupérer le **bridge** LCP → installation → facility.

### Étape 5
Créer la table `gold_facility_year_environment_profile`.

### Étape 6
Ajouter les enrichissements eau, transferts, benchmark secteur.

### Étape 7
Faire les visualisations et raconter une histoire analytique claire.



## 14. Ce que je veux que tu retiennes

Tu n'as pas raté ton projet.  
Tu as juste rencontré un **vrai problème de modélisation**, qui est normal.

En fait, ton projet devient meilleur si tu expliques clairement :

- que les tables sont à des grains différents ;
- que tu as construit une stratégie de rapprochement propre ;
- que ton indicateur principal est un **proxy** ;
- et que tu enrichis ensuite le profil environnemental des sites.

Si tu fais ça proprement, ton projet devient beaucoup plus crédible.



## 15. Mini checklist avant de continuer

Avant d'aller plus loin, vérifie que tu peux répondre à ces questions :

- [ ] Quel est le grain de chaque table ?
- [ ] Quel est le grain de ma table finale ?
- [ ] Quelle est ma vraie clé de jointure ?
- [ ] Ai-je un bridge documenté ?
- [ ] Mon indicateur est-il exact ou proxy ?
- [ ] Ai-je aligné la période d'analyse ?
- [ ] Ai-je évité les duplications ?

Si tu peux répondre à ça, tu es sur une bonne voie.



## 16. Dernier conseil

Pour **toi**, je te conseille de viser un projet **propre et clair**, pas un projet trop large.

Donc :
- Belgique
- 2016–2024
- CO2 facility
- énergie LCP
- enrichissement eau + transferts
- benchmark secteur

C'est déjà un très bon projet de data engineering.
